# SVCM — ITI-96 — Récupération CodeSystem

**Profil** : IHE ITI SVCM
**Acteur testé** : SVCM-Terminology Consumer
**Transaction** : SVCM-ITI-96
**Référence Gazelle** : https://interop.esante.gouv.fr/tm/testing/testsDefinition/viewTestPage.seam?id=12207&cid=48534

Récupérer le CodeSystem `TRE-R02-SecteurActivite` par URL canonique, puis par `fullUrl`.

## Configuration

In [ ]:
import requests
import json
import time
from urllib.parse import quote

FHIR_BASE = "https://smt.esante.gouv.fr/fhir"
API_KEY   = "ZiaoDxF8Je0CfBNzu..."   # Remplacer par la clé API complète

HTTP_RETRIES = 3
HTTP_BACKOFF = 2

HEADERS_JSON = {
    "Accept": "application/fhir+json",
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}

def http_get(url, headers=None):
    if headers is None:
        headers = HEADERS_JSON
    for attempt in range(1, HTTP_RETRIES + 1):
        try:
            print(f"  → GET {url}")
            r = requests.get(url, headers=headers, allow_redirects=True)
            if r.status_code == 200:
                return r
            raise Exception(f"HTTP {r.status_code}: {r.text[:200]}")
        except Exception as e:
            print(f"  ✗ tentative {attempt}/{HTTP_RETRIES} : {e}")
            if attempt == HTTP_RETRIES:
                raise
            time.sleep(HTTP_BACKOFF ** attempt)

def print_keys(data, *keys):
    subset = {k: data.get(k) for k in keys if k in data}
    print(json.dumps(subset, indent=2, ensure_ascii=False))

print("Configuration OK — prêt à exécuter les étapes.")


---
## Étapes 0–30 — Récupération du CodeSystem TRE-R02-SecteurActivite

**Requête** : `GET /fhir/CodeSystem?url=https://mos.esante.gouv.fr/NOS/TRE_R02-SecteurActivite/FHIR/TRE-R02-SecteurActivite&_format=json`
**Objectif** : Localiser le CodeSystem par URL canonique, extraire le `fullUrl`, puis le récupérer directement.

In [ ]:
# Étape 0  — Construction de la requête
# Étape 10 — TRANSACTION : GET CodeSystem par URL canonique
TRE_R02_URL = "https://mos.esante.gouv.fr/NOS/TRE_R02-SecteurActivite/FHIR/TRE-R02-SecteurActivite"
url_search = f"{FHIR_BASE}/CodeSystem?url={quote(TRE_R02_URL)}&_format=json"

r_search = http_get(url_search)
bundle = r_search.json()

# Étape 20 — Réception et intégration
entries = bundle.get("entry", [])
assert len(entries) > 0, f"Aucune entrée trouvée pour {TRE_R02_URL}"

cs_entry   = entries[0]
full_url   = cs_entry.get("fullUrl", "")
cs_summary = cs_entry.get("resource", {})

print(f"Entrées trouvées : {len(entries)}")
print(f"fullUrl          : {full_url}")
print(f"  name    : {cs_summary.get('name')}")
print(f"  version : {cs_summary.get('version')}")
print(f"  content : {cs_summary.get('content')}")
print(f"  status  : {cs_summary.get('status')}")

# Étape 30 — PREUVE : GET par fullUrl, affichage des champs clés
print(f"\n[PREUVE étape 30] GET par fullUrl : {full_url}")
r_direct = http_get(f"{full_url}?_format=json")
cs = r_direct.json()

print_keys(cs, "resourceType", "id", "url", "version", "name", "title", "status", "content")
concepts = cs.get("concept", [])
print(f"\nNombre de concepts : {len(concepts)}")
if concepts:
    print("Premiers codes :")
    for c in concepts[:10]:
        print(f"  {c.get('code')} — {c.get('display')}")
    if len(concepts) > 10:
        print(f"  ... ({len(concepts) - 10} autres)")
